In [1]:
import pandas as pd
import numpy as np
import os, sys
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
import tkinter as tk
from tkinter import messagebox

# Load and prepare data
df = pd.read_csv("GitLab app_vulnerabilities_2025-10-22T1757.csv")
if 'Vulnerability' not in df.columns or 'Status' not in df.columns:
    sys.exit("Error: Dataset must contain 'Vulnerability' and 'Status' columns.")
df.dropna(subset=['Vulnerability', 'Status'], inplace=True)

X_train, X_test, y_train, y_test = train_test_split(
    df['Vulnerability'], df['Status'], test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    max_features=5000
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)

print("\nModel Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))




Model Accuracy: 0.7341730195558502

Classification Report:
               precision    recall  f1-score   support

    detected       0.74      0.88      0.80      1599
   dismissed       0.72      0.62      0.67      1285
    resolved       0.56      0.08      0.13       133

    accuracy                           0.73      3017
   macro avg       0.67      0.53      0.54      3017
weighted avg       0.73      0.73      0.72      3017



User input for vulnerability descriptions

In [48]:
def predict_vulnerability():
    user_text = entry.get()
    if user_text.lower() == 'x':
        root.destroy()
        return
    if not user_text.strip():
        messagebox.showwarning("Input Error", "Please enter a vulnerability description.")
        return

    # Transform the input text
    new_tfidf = vectorizer.transform([user_text])

    # Predict each field independently
    predicted_tool = tool_model.predict(new_tfidf)[0]
    predicted_scanner = scanner_model.predict(new_tfidf)[0]
    predicted_status = status_model.predict(new_tfidf)[0]
    predicted_dismissal = dismissal_model.predict(new_tfidf)[0]

    # Display results
    result_label.config(
        text=(
            f"Vulnerability: {user_text}\n"
            f"Predicted Tool: {predicted_tool}\n"
            f"Predicted Scanner Name: {predicted_scanner}\n"
            f"Predicted Status: {predicted_status}\n"
            f"Predicted Dismissal Reason: {predicted_dismissal}"
        )
    )

Predictor GUI

In [49]:
root = tk.Tk()
root.title("Vulnerability Triage ML App")

entry_label = tk.Label(root, text="Enter vulnerability description (or 'X' to exit):")
entry_label.pack(pady=5)

entry = tk.Entry(root, width=60)
entry.pack(pady=5)

predict_button = tk.Button(root, text="Predict", command=predict_vulnerability)
predict_button.pack(pady=5)

result_label = tk.Label(root, text="", fg="blue")
result_label.pack(pady=10)

root.mainloop()

Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.2544.0_x64__qbz5n2kfra8p0\Lib\tkinter\__init__.py", line 2074, in __call__
    return self.func(*args)
           ~~~~~~~~~^^^^^^^
  File "C:\Temp\1\ipykernel_28580\4219700113.py", line 14, in predict_vulnerability
    predicted_tool = tool_model.predict(new_tfidf)[0]
                     ^^^^^^^^^^
NameError: name 'tool_model' is not defined
